<a href="https://colab.research.google.com/github/mitalidaduria/.github/blob/main/src_data_gen_gateway_simulator_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [1]:
!pip install -q pyspark==3.5.0 delta-spark==3.1.0 pydantic fastapi uvicorn requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 10.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.


In [13]:
import random
from datetime import datetime, timezone
import uuid

CANONICAL_FAILURES = [
    "DUPLICATE", "TIMEOUT", "NSF", "GATEWAY_REJECT",
    "AUTH_FAIL", "SETTLEMENT_DISCREPANCY", "ROUTING_ERROR", "SYSTEM_ERROR"
]

def generate_payment_events(num_records=10):
    gateways = ["razorpay", "stripe", "payu"]
    currencies = ["INR", "USD", "EUR"]
    merchants = ["m_merch_101", "m_merch_102", "m_merch_103", "m_merch_104"]

    events = []
    for _ in range(num_records):
        status = random.choices(["SUCCESS", "FAILED"], weights=[0.82, 0.18])[0]
        failure_reason = None if status == "SUCCESS" else random.choice(CANONICAL_FAILURES)

        event = {
            "transaction_id": str(uuid.uuid4()),
            "gateway": random.choice(gateways),
            "amount": round(random.uniform(10.0, 50000.0), 2),
            "currency": random.choice(currencies),
            "status": status,
            "failure_reason": failure_reason,
            "merchant_id": random.choice(merchants),
            "timestamp": datetime.now(timezone.utc).isoformat()
        }
        events.append(event)
    return events

# Test the simulator
sample_events = generate_payment_events(3)
for ev in sample_events:
    print(ev)

{'transaction_id': '8a00fd47-9cf2-40ac-8b4b-02865e018ae4', 'gateway': 'razorpay', 'amount': 14671.27, 'currency': 'USD', 'status': 'SUCCESS', 'failure_reason': None, 'merchant_id': 'm_merch_102', 'timestamp': '2026-10-01T08:16:41.239369+00:00'}
{'transaction_id': '790092b1-a9c0-4735-8dd0-da2c40354f7e', 'gateway': 'payu', 'amount': 6623.31, 'currency': 'USD', 'status': 'SUCCESS', 'failure_reason': None, 'merchant_id': 'm_merch_104', 'timestamp': '2026-10-01T08:16:41.239436+00:00'}
{'transaction_id': 'c7d5dbc1-3d62-45b6-8177-5bdae4142b63', 'gateway': 'stripe', 'amount': 17991.91, 'currency': 'INR', 'status': 'SUCCESS', 'failure_reason': None, 'merchant_id': 'm_merch_104', 'timestamp': '2026-10-01T08:16:41.239474+00:00'}


In [15]:
from fastapi import FastAPI, Header, status
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import Optional
import uuid

# 1. Define FastAPI App
app = FastAPI(
    title="Payment Ingestion Gateway",
    version="1.0",
    description="Asynchronous payment ingestion gateway adhering to production DQ contracts."
)

class PaymentEvent(BaseModel):
    transaction_id: str
    gateway: str
    amount: float = Field(..., gt=0, description="Transaction amount in major currency units")
    currency: str
    status: str
    failure_reason: Optional[str] = None
    merchant_id: str
    timestamp: str

# In-memory buffer to temporarily store received events
INGESTED_BUFFER = []

@app.post("/v1/events/payment", status_code=status.HTTP_202_ACCEPTED)
async def ingest_payment(
    event: PaymentEvent,
    x_idempotency_key: Optional[str] = Header(None, description="Client-provided idempotency key")
):
    event_dict = event.dict()
    event_dict["idempotency_key"] = x_idempotency_key
    INGESTED_BUFFER.append(event_dict)

    return {
        "status": "ACCEPTED",
        "message": "Event queued successfully for pipeline ingestion",
        "transaction_id": event.transaction_id,
        "idempotency_key": x_idempotency_key
    }

# 2. Use FastAPI's built-in TestClient (bypasses uvicorn background threading issues in Colab)
client = TestClient(app)

# Test the endpoint using a payload from our simulator
test_payload = generate_payment_events(1)[0]
test_headers = {"X-Idempotency-Key": str(uuid.uuid4())}

response = client.post("/v1/events/payment", json=test_payload, headers=test_headers)

print("--- Test Ingestion via TestClient ---")
print("Response Status Code:", response.status_code)
print("Response Body:", response.json())
print("Total events currently in buffer:", len(INGESTED_BUFFER))

--- Test Ingestion via TestClient ---
Response Status Code: 202
Response Body: {'status': 'ACCEPTED', 'message': 'Event queued successfully for pipeline ingestion', 'transaction_id': '8f7430e0-cd9c-41ba-93c2-52f37f1cae40', 'idempotency_key': 'ea935273-4f77-47b8-9190-4d30662962cf'}
Total events currently in buffer: 1


/tmp/ipykernel_3087/1428537229.py:32: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  event_dict = event.dict()


In [24]:
from pyspark.sql import SparkSession

# Re-initialize a clean Spark session without Delta catalog overrides
spark.stop()
spark = SparkSession.builder \
    .appName("PaymentPipeline-Clean") \
    .config("spark.sql.catalogImplementation", "in-memory") \
    .getOrCreate()

print("Clean Spark session initialized successfully!")

Clean Spark session initialized successfully!


In [27]:
import pandas as pd
from datetime import datetime, timezone

# 1. Load buffer into a pandas DataFrame (Bronze Layer representation)
bronze_df = pd.DataFrame(INGESTED_BUFFER)
bronze_df["ingest_timestamp"] = datetime.now(timezone.utc).isoformat()
bronze_df["partition_date"] = pd.to_datetime(bronze_df["timestamp"]).dt.date

print(f"Bronze Layer Loaded: {len(bronze_df)} records.")

# 2. Silver Transformation: Deduplication (keeping latest record via idempotency/timestamp)
dedup_df = bronze_df.sort_values(by="ingest_timestamp", ascending=False).drop_duplicates(subset=["transaction_id"], keep="first")

# 3. Type Normalization (Amount to minor units × 100) & Canonical Failure Classification
dedup_df["amount_minor_units"] = (dedup_df["amount"] * 100).astype("int64")

def classify_failure(row):
    if row["status"] == "SUCCESS":
        return None
    if pd.isna(row["failure_reason"]):
        return "SYSTEM_ERROR"
    return row["failure_reason"]

dedup_df["failure_category_cleaned"] = dedup_df.apply(classify_failure, axis=1)

# 4. Handle Critical Null Fields -> Route to DQ Exception View instead of dropping silently
critical_fields = ["transaction_id", "gateway", "amount"]
clean_mask = dedup_df[critical_fields].notna().all(axis=1)

clean_silver_df = dedup_df[clean_mask]
dq_exceptions_df = dedup_df[~clean_mask]

print(f"Silver Clean Records Count: {len(clean_silver_df)}")
print(f"DQ Exceptions Flagged Count: {len(dq_exceptions_df)}")

print("\n--- Clean Silver Sample ---")
print(clean_silver_df[["transaction_id", "gateway", "amount_minor_units", "status", "failure_category_cleaned"]].head(3))

Bronze Layer Loaded: 1 records.
Silver Clean Records Count: 1
DQ Exceptions Flagged Count: 0

--- Clean Silver Sample ---
                         transaction_id gateway  amount_minor_units  status  \
0  8f7430e0-cd9c-41ba-93c2-52f37f1cae40  stripe              346271  FAILED   

  failure_category_cleaned  
0                AUTH_FAIL  


In [28]:
# === WEEK 3: 3-Layer Data Quality & Governance Framework ===

print("=== LAYER 2: Business Rule Enforcement (Silver Tier) ===")
# Apply rule checks with severity tags (P1: Hard block, P2: Soft warning)
clean_silver_df["rule_negative_amount"] = clean_silver_df["amount_minor_units"].apply(lambda x: "P1_BLOCK" if x <= 0 else "PASS")
clean_silver_df["rule_invalid_currency"] = clean_silver_df["currency"].apply(lambda x: "P2_WARN" if x not in ["INR", "USD", "EUR"] else "PASS")
clean_silver_df["rule_missing_failure_reason"] = clean_silver_df.apply(
    lambda row: "P1_BLOCK" if row["status"] == "FAILED" and pd.isna(row["failure_reason"]) else "PASS", axis=1
)

# Separate business exceptions (P1 hard blocks) from valid records
business_exceptions_df = clean_silver_df[
    (clean_silver_df["rule_negative_amount"] == "P1_BLOCK") |
    (clean_silver_df["rule_missing_failure_reason"] == "P1_BLOCK")
]
valid_silver_df = clean_silver_df[
    (clean_silver_df["rule_negative_amount"] != "P1_BLOCK") &
    (clean_silver_df["rule_missing_failure_reason"] != "P1_BLOCK")
]

print(f"Layer 2 Check: {len(business_exceptions_df)} records flagged with P1 Business Rule blocks.")
print(f"Layer 2 Check: {len(valid_silver_df)} records successfully passed all business rules.")

print("\n=== LAYER 3: Gold Aggregations & STP Calculation ===")
# Compute Straight-Through Processing (STP) rate and volume totals by gateway & currency
gold_aggregates = valid_silver_df.groupby(["gateway", "currency", "status"]).agg(
    total_transactions=("transaction_id", "count"),
    total_amount_minor=("amount_minor_units", "sum")
).reset_index()

print(gold_aggregates.to_string(index=False))

=== LAYER 2: Business Rule Enforcement (Silver Tier) ===
Layer 2 Check: 0 records flagged with P1 Business Rule blocks.
Layer 2 Check: 1 records successfully passed all business rules.

=== LAYER 3: Gold Aggregations & STP Calculation ===
gateway currency status  total_transactions  total_amount_minor
 stripe      INR FAILED                   1              346271
